# 04 · Exploratory: overlap-free validation (Kaggle GPU, ~10-15 min)

Runs `benchmarks/clean_val_eval.py`: Turkish BPB of tr030, tr030_seed1, the two x3 controls and the Turkish-only control on the full validation set and on three subsets without documents that overlap the repeated training windows. Not part of the analysis plan. Results are uploaded to the runs repo under `analysis/`.

**Setup:** Settings → Accelerator: *GPU T4* (one is enough), Internet: *On*. Add-ons → Secrets: `HF_TOKEN`. Run after the x3 runs have finished.

In [ ]:
HF_USER = "DeerKaraca"
DATA_REPO = f"{HF_USER}/bilingual-slm-data"
RUNS_REPO = f"{HF_USER}/bilingual-slm-runs"
RUNS = ["tr030", "tr030_seed1", "tr_only_control_x3", "tr_only_control_x3_seed1", "tr_only_control"]

In [ ]:
!nvidia-smi --query-gpu=name --format=csv
!git clone -q https://github.com/DeerKaraca/bilingual-slm.git
%cd bilingual-slm
!git log --oneline -1
!pip install -q -e ".[data,train]"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, snapshot_download

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
snapshot_download(DATA_REPO, repo_type="dataset", allow_patterns=["tokens/tr/*"], local_dir="data")
for run in RUNS:
    hf_hub_download(RUNS_REPO, f"{run}/final.pt", local_dir="runs")

In [ ]:
!python benchmarks/clean_val_eval.py --device cuda --out /kaggle/working/clean_val

In [ ]:
api = HfApi()
for ext in ("md", "json"):
    api.upload_file(path_or_fileobj=f"/kaggle/working/clean_val.{ext}", path_in_repo=f"analysis/clean_val.{ext}", repo_id=RUNS_REPO)
print("uploaded to", RUNS_REPO)